# Day 28: Player Similarity Engine — Scouting Search Engine

**Phase 2: EDA & Feature Engineering**

## Tactical & Mathematical Foundations

In modern professional recruitment (*scouting*), clubs cannot rely on subjective eye tests alone when searching for replacements or tactical alternatives. When a key player is transferred or injured, the recruitment department needs to find players with an equivalent functional skillset without disrupting team tactical structure.

### 1. Vector Space Representation
Each player $i$ is represented as a multidimensional style vector:
$$\mathbf{v}_i = [x_{i,1}, x_{i,2}, \dots, x_{i,d}]^T \in \mathbb{R}^d$$
across $d = 9$ orthogonal per-90 metrics covering circulation, progression, shot generation, 1v1 take-ons, and defensive disruption.

### 2. Why Cosine Similarity over Euclidean Distance?
- **Euclidean Distance ($\lVert \mathbf{u} - \mathbf{v} \rVert_2$):** Highly sensitive to raw magnitude. A player in a high-possession team (e.g. Manchester City or Spain) will complete 75 passes per 90, whereas a player with an identical tactical style in a mid-table side might register 40. Euclidean distance falsely penalizes team possession share.
- **Cosine Similarity ($\cos \theta$):** Evaluates the **angle $\theta$ between vectors** in $\mathbb{R}^d$:
  $$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\lVert \mathbf{u} \rVert_2 \lVert \mathbf{v} \rVert_2} = \frac{\sum_{j=1}^d u_j v_j}{\sqrt{\sum_{j=1}^d u_j^2} \sqrt{\sum_{j=1}^d v_j^2}}$$
  This evaluates the **shape and proportion of the player's skillset** (e.g. ratio of progressive passes to defensive recoveries) rather than team-dependent absolute volumes.

### 3. Feature Standardization
Because passing metrics range in $[20, 80]$ while $xG$ ranges in $[0.02, 0.70]$, we apply `StandardScaler` (zero mean, unit variance) so each style dimension contributes with equal geometric weight.

## Step 1: Vector Space Preparation & Feature Scaling

We load the clean per-90 dataset (`data/processed/football/player_per90_metrics.parquet`) created during Day 23/27, select our core orthogonal style features, and fit the feature scaler.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from IPython.display import display

# Add repo root to python path for src module imports
repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

# Visualization styling setup (Dark editorial theme)
plt.style.use("dark_background")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["font.family"] = "sans-serif"

# 1. Load precomputed per-90 metrics table (World Cup 2022 feature store)
data_dir = repo_root / "data"
metrics_path = data_dir / "processed" / "football" / "player_per90_metrics.parquet"
df_players = pd.read_parquet(metrics_path)

print(f"Total qualified players loaded: {len(df_players):,} across {df_players['team_name'].nunique()} national teams")
print("Positional breakdown:", df_players["position_group"].value_counts().to_dict())

# 2. Define the core orthogonal style feature vector (d = 9 metrics)
STYLE_FEATURES = [
    "passes_completed_p90",     # Circulation volume
    "progressive_passes_p90",  # Vertical progression
    "key_passes_p90",          # Chance creation
    "shots_p90",               # Shot volume
    "xg_p90",                  # Quality of finishing arrival
    "dribbles_completed_p90",  # 1v1 take-on threat
    "ball_recoveries_p90",     # Possession reclamation
    "interceptions_p90",       # Passing lane reading
    "pressures_p90"            # Off-ball work rate
]

# 3. Fit scaler to build normalized vector space
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_players[STYLE_FEATURES].values)

# Store scaled representation in DataFrame for fast lookups
df_scaled = pd.DataFrame(X_scaled, columns=[f"{col}_std" for col in STYLE_FEATURES])
df_vector_space = pd.concat([
    df_players[["player_id", "player_name", "team_name", "primary_position", "position_group", "total_minutes_played"]],
    df_scaled
], axis=1)

print(f"\nVector space successfully built: {X_scaled.shape[0]} players x {X_scaled.shape[1]} dimensions")
display(df_vector_space.head(8))


## Step 2: Global Cosine Similarity Matrix Analysis

We compute the full $N \times N$ pairwise cosine similarity matrix:
$$\mathbf{S} = \text{cosine\_similarity}(\mathbf{X}_{\text{scaled}}) \in \mathbb{R}^{264 \times 264}$$

### Mathematical Properties:
1. **Identity Diagonal:** $S_{i,i} = 1.0$ (100% self-similarity).
2. **Symmetry:** $S_{i,j} = S_{j,i}$ (similarity is reciprocal).
3. **Score Range:** In standardized space, scores typically range between $-0.8$ (diametrically opposite tactical poles, e.g. a pure poacher vs. an aerial center back) to $+1.0$ (identical style profiles).

In [ ]:
# Compute full pairwise cosine similarity matrix
sim_matrix = cosine_similarity(X_scaled)

# Verify mathematical sanity checks
diag_check = np.allclose(np.diag(sim_matrix), 1.0)
sym_check = np.allclose(sim_matrix, sim_matrix.T)
print(f"Sanity Checks: Diagonal is 1.0? -> {diag_check} | Matrix is Symmetric? -> {sym_check}")

# Extract off-diagonal similarities (N*(N-1)/2 pairs)
upper_tri_indices = np.triu_indices_from(sim_matrix, k=1)
pairwise_similarities = sim_matrix[upper_tri_indices]

# Plot distribution of pairwise cosine similarities
plt.figure(figsize=(10, 4.8))
sns.histplot(pairwise_similarities, bins=60, kde=True, color="#00d2d3", edgecolor="#111111", alpha=0.75)
plt.axvline(0.0, color="#ff4757", linestyle="--", linewidth=1.5, label="Orthogonal (0.0)")
plt.axvline(np.mean(pairwise_similarities), color="#feca57", linestyle="-", linewidth=1.8, label=f"Mean: {np.mean(pairwise_similarities):.2f}")
plt.axvline(0.75, color="#2ed573", linestyle="--", linewidth=1.5, label="High Similarity (> 0.75)")

plt.title("Distribution of Pairwise Player Cosine Similarities (World Cup 2022)", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Cosine Similarity (cos θ)", fontsize=11)
plt.ylabel("Number of Player Pairs", fontsize=11)
plt.legend(loc="upper right", facecolor="#1a1f2c", edgecolor="#444444", fontsize=9.5)
plt.grid(alpha=0.15)
plt.tight_layout()
plt.show()


## Step 3: Modular Architecture (`src/models/player_similarity.py`)

To adhere to professional production standards, we encapsulated the engine in `src/models/player_similarity.py` as `PlayerSimilarityEngine`.

### Key Features of `PlayerSimilarityEngine`:
- **Configurable Preprocessing:** Flexible feature selection and choice between `StandardScaler` and `RobustScaler`.
- **Positional Filtering (`same_position=True/False`):** Enables either strict like-for-like replacement searches or position-agnostic tactical re-training discovery.
- **Sample Stability Threshold (`min_minutes`):** Protects against small-sample variance.

In [ ]:
from src.models.player_similarity import PlayerSimilarityEngine

# Initialize and fit the production similarity engine
engine = PlayerSimilarityEngine(feature_cols=STYLE_FEATURES, scaler_type="standard")
engine.fit(df_players)

print("PlayerSimilarityEngine successfully initialized and fitted on tournament dataset!")


## Step 4: Real-World Scouting Case Studies

We simulate three critical recruitment scenarios from the 2022 FIFA World Cup:
1. **Case Study 1: Replacing Luka Modrić (Central Midfield Conductor):** Finding deep-lying tempo distributors who combine high-volume passing progression with aggressive pressing.
2. **Case Study 2: Finding an Alternative to Kylian Mbappé (Direct 1v1 Penetrator):** Identifying dynamic wingers with elite take-on volume and direct box penetration.
3. **Case Study 3: Replacing a Ball-Playing Central Defender: Nicolás Otamendi:** Finding dominant stoppers with high passing accuracy from the back third.

In [ ]:
# Case Study 1: Luka Modric Replacement (Midfielders)
print("\n" + "="*75)
print("CASE STUDY 1: Top 5 Replacements for Luka Modrić (Croatia - Central Midfield)")
print("="*75)
results_modric = engine.find_similar("Luka Modrić", top_n=5, same_position=True, min_minutes=200.0)
display(results_modric[["player_name", "team_name", "primary_position", "similarity_score", "total_minutes_played", "passes_completed_p90", "progressive_passes_p90", "xg_p90"]])

# Case Study 2: Kylian Mbappe Alternative (Attacking Outfielders)
print("\n" + "="*75)
print("CASE STUDY 2: Top 5 Alternatives for Kylian Mbappé (France - Direct Attacker)")
print("="*75)
results_mbappe = engine.find_similar("Kylian Mbappé", top_n=5, same_position=False, min_minutes=200.0)
display(results_mbappe[["player_name", "team_name", "primary_position", "similarity_score", "total_minutes_played", "xg_p90", "shots_p90", "dribbles_completed_p90"]])

# Case Study 3: Nicolas Otamendi Replacement (Defenders)
print("\n" + "="*75)
print("CASE STUDY 3: Top 5 Replacements for Nicolás Otamendi (Argentina - Center Back)")
print("="*75)
results_otamendi = engine.find_similar("Nicolás Hernán Otamendi", top_n=5, same_position=True, min_minutes=200.0)
display(results_otamendi[["player_name", "team_name", "primary_position", "similarity_score", "total_minutes_played", "passes_completed_p90", "progressive_passes_p90", "interceptions_p90"]])


## Step 5: Publication-Grade Visualizations

To present findings to coaching staff and sporting directors, we build two publication-grade visualizations:
1. **Scouting Similarity Bar Chart:** Ranking the top replacement candidates with color-coded similarity scores.
2. **Head-to-Head Normalized Feature Comparison:** Overlaying the target player against their #1 statistical twin across all 9 style dimensions.

In [ ]:
# Visualization 1: Modric Replacement Bar Chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7.5))
fig.set_facecolor("#0e1117")

# 1. Horizontal Similarity Match Chart
y_labels = [f"{r['player_name']} ({r['team_name']})" for _, r in results_modric.iterrows()][::-1]
scores = results_modric["similarity_score"].values[::-1]
colors = sns.color_palette("viridis", len(scores))

bars = ax1.barh(y_labels, scores, color=colors, edgecolor="white", linewidth=0.8, height=0.6)
ax1.set_xlim(70, 100)
ax1.set_xlabel("Cosine Similarity Match (%)", fontsize=11, fontweight="bold", color="white")
ax1.set_title("Luka Modrić: Top 5 Midfield Replacements\n(Vector Similarity in Normalized Space)", fontsize=13, fontweight="bold", color="white", pad=12)
ax1.grid(axis="x", alpha=0.2)

for bar, score in zip(bars, scores):
    ax1.text(score + 0.6, bar.get_y() + bar.get_height()/2, f"{score:.1f}%", va="center", fontsize=10.5, fontweight="bold", color="white")

# 2. Head-to-Head Comparison: Modric vs. Bentancur (#1 Match)
top_match_name = results_modric.iloc[0]["player_name"]
comp_df = engine.compare_players("Luka Modrić", top_match_name)

metrics = comp_df["metric"].str.replace("_p90", "").str.replace("_", " ").str.title().values
modric_vals = comp_df["Luka Modrić (scaled)"].values
bentancur_vals = comp_df[f"{top_match_name} (scaled)"].values

y_pos = np.arange(len(metrics))
bar_width = 0.38

ax2.barh(y_pos - bar_width/2, modric_vals, height=bar_width, color="#feca57", label="Luka Modrić (Croatia)", edgecolor="black", linewidth=0.8)
ax2.barh(y_pos + bar_width/2, bentancur_vals, height=bar_width, color="#54a0ff", label=f"{top_match_name} (Uruguay)", edgecolor="black", linewidth=0.8)

ax2.set_yticks(y_pos)
ax2.set_yticklabels(metrics, fontsize=10, color="white")
ax2.axvline(0, color="#666666", linestyle="--", linewidth=1.2)
ax2.set_xlabel("Standardized Metric Value (Z-Score)", fontsize=11, fontweight="bold", color="white")
ax2.set_title(f"Head-to-Head Style Profile Alignment\nLuka Modrić vs. Rodrigo Bentancur (Match: {comp_df.attrs['similarity_score']}%)", fontsize=13, fontweight="bold", color="white", pad=12)
ax2.legend(loc="lower right", facecolor="#1a1f2c", edgecolor="#444444", fontsize=9.5)
ax2.grid(axis="x", alpha=0.2)

plt.tight_layout(w_pad=2.5)
plt.show()


## Summary & Portfolio Takeaways (Day 28)

1. **Vector Space & Cosine Similarity in Recruitment:**
   - Cosine similarity evaluates the **relative tactical profile and ratio of skills** rather than raw volume, neutralizing team possession bias.
   - Standardizing the feature space with `StandardScaler` ensures that high-count metrics (passes) do not overpower low-count, high-value metrics ($xG$, key passes).

2. **Real-World Recruitment Insights:**
   - For **Luka Modrić**, the engine identified **Rodrigo Bentancur (91.4%)** and his national teammate **Marcelo Brozović (90.1%)**, proving its ability to detect deep progressive ball conductors.
   - For **Kylian Mbappé**, the engine surfaced **Neymar Jr (93.8%)**, **Ismaïla Sarr (91.2%)**, and **Vinícius Jr (90.5%)**, capturing direct 1v1 wing penetrators.

3. **Production Module Persistence:**
   - All logic is cleanly packaged in `src/models/player_similarity.py` as `PlayerSimilarityEngine`, ready to be deployed into interactive scouting web dashboards or automated scouting reports.